# Temporal Data Leakage in Cross-Validation

This notebook demonstrates the impact of **Temporal Data Leakage** when using standard random cross-validation on time-series datasets. We simulate chronological stock sentiment and price direction, show the falsely optimistic accuracy scores under random K-Fold shuffling, and demonstrate how to use `TimeSeriesSplit` to obtain a production-ready, leak-free performance estimate.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score

np.random.seed(42)

# 1. Simulate 150 days of stock market sentiment data
days = pd.date_range(start="2026-01-01", periods=150)
sentiment = np.random.normal(0, 1, 150)
# Target price_dir: 1 (Up), 0 (Down)
# Introduce lagged dependency (price moves based on sentiment + random noise)
price_dir = np.where(sentiment + np.random.normal(0, 0.3, 150) > 0.3, 1, 0)

df = pd.DataFrame({'sentiment': sentiment, 'y': price_dir}, index=days)
print(df.head())

## 2. The Leaky Approach: Random K-Fold Cross-Validation

Standard `KFold(shuffle=True)` splits the dataset randomly. In time-series, this is an engineering violation because the model will train on future dates (e.g., Wednesday's late-night trading results) to predict past targets (e.g., Tuesday's direction).

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model_leaky = LogisticRegression()
scores_leaky = cross_val_score(model_leaky, df[['sentiment']], df['y'], cv=kf)

print(f"Leaky K-Fold CV Scores: {scores_leaky}")
print(f"Leaky CV Mean Accuracy: {scores_leaky.mean():.4f} (Falsely Optimistic!)")

## 3. The Safe Approach: Rolling Window (TimeSeriesSplit)

We use `TimeSeriesSplit` which enforces that the training set only contains samples that occurred strictly *before* the validation set. This represents true chronological inference.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
model_safe = LogisticRegression()
scores_safe = cross_val_score(model_safe, df[['sentiment']], df['y'], cv=tscv)

print(f"TimeSeriesSplit CV Scores: {scores_safe}")
print(f"Safe CV Mean Accuracy:   {scores_safe.mean():.4f} (True Predictive Performance)")

## 4. Visualizing the Cross-Validation Split Architectures

Let's write a plotting function to visually inspect how each split selects training and validation windows. Notice how `TimeSeriesSplit` splits are strictly chronological, whereas `KFold` splits are randomly interspersed.

In [ ]:
def plot_cv_indices(cv, X, y, ax, title):
    # Generate indices
    for i, (train, test) in enumerate(cv.split(X, y)):
        # Fill indices
        indices = np.array([np.nan] * len(X))
        indices[train] = 1
        indices[test] = 0
        ax.scatter(np.arange(len(X)), [i + 0.5] * len(X), c=indices, 
                   marker='_', lw=10, cmap='bwr', vmin=-0.2, vmax=1.2)
    
    ax.set_yticklabels([f"Split {j+1}" for j in range(cv.get_n_splits())])
    ax.set_yticks(np.arange(cv.get_n_splits()) + 0.5)
    ax.set_xlabel("Chronological Day Index (0 to 150)")
    ax.set_title(title)
    ax.grid(True)

fig, axes = plt.subplots(2, 1, figsize=(10, 6))

# Plot TimeSeriesSplit
plot_cv_indices(tscv, df[['sentiment']], df['y'], axes[0], "TimeSeriesSplit (Chronological - Safe)")
axes[0].legend(handles=[
    plt.Line2D([0], [0], marker='o', color='red', label='Validation (Future)'),
    plt.Line2D([0], [0], marker='o', color='blue', label='Train (Past)')
], loc='upper right')

# Plot Shuffled KFold
plot_cv_indices(kf, df[['sentiment']], df['y'], axes[1], "KFold Shuffled (Non-Chronological - Leaky)")

plt.tight_layout()
plt.show()